In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

print("Iniciando Script 2: Limpieza, Rangos Dinámicos y Separación de Tablas...")

# 1. CARGA DEL ARCHIVO CRUDO
df = pd.read_parquet("data/raw/sima_raw.parquet")
cols_medicion = ['CO', 'NO', 'NO2', 'NOX', 'O3', 'PM10', 'PM2.5', 'PRS', 'RAINF', 'RH', 'SO2', 'SR', 'TOUT', 'WSR', 'WDR']

# 2. DESTRUCCIÓN DE BANDERAS
print("Eliminando banderas de invalidación y ajustando tipos...")
for col in cols_medicion:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

df['Year'] = df['Date'].dt.year

# 3. FILTRO DE RANGOS FÍSICOS (Según PDF del SIMA)
print("Aplicando rangos operativos del fabricante y del SIMA por año...")
# Diccionario con los límites permitidos (min, max) año por año
rangos = {
    2020: {'PM10': (0, 800), 'PM2.5': (0, 205.94), 'O3': (0, 153), 'WSR': (0, 75), 'TOUT': (0, 41), 'PRS': (690, 750), 'CO': (0, 20), 'NO': (0, 500), 'NO2': (0, 200), 'NOX': (0, 500), 'SO2': (0, 200), 'SR': (0, 1)},
    2021: {'PM10': (0, 800), 'PM2.5': (0, 325), 'O3': (0, 175), 'WSR': (0, 40), 'TOUT': (-6.5, 45), 'PRS': (690, 740), 'CO': (0, 10), 'NO': (0, 350), 'NO2': (0, 100), 'NOX': (0, 400), 'SO2': (0, 300), 'SR': (0, 1)},
    2022: {'PM10': (0, 999), 'PM2.5': (0, 450), 'O3': (0, 160), 'WSR': (0, 35), 'TOUT': (-5, 45), 'PRS': (700, 740), 'CO': (0, 8), 'NO': (0, 400), 'NO2': (0, 175), 'NOX': (0, 420), 'SO2': (0, 200), 'SR': (0, 1.25)},
    2023: {'PM10': (0, 900), 'PM2.5': (0, 800), 'O3': (0, 175), 'WSR': (0, 40), 'TOUT': (0, 45), 'PRS': (690, 740), 'CO': (0, 14), 'NO': (0, 500), 'NO2': (0, 175), 'NOX': (0, 500), 'SO2': (0, 250), 'SR': (0, 1)},
    2024: {'PM10': (0, 999), 'PM2.5': (0, 999), 'O3': (0, 180), 'WSR': (0, 38), 'TOUT': (-4, 45.5), 'PRS': (687.5, 740), 'CO': (0, 18), 'NO': (0, 400), 'NO2': (0, 130), 'NOX': (0, 500), 'SO2': (0, 150), 'SR': (0, 1.26)},
    2025: {'PM10': (0, 820), 'PM2.5': (0, 350), 'O3': (0, 185), 'WSR': (0, 40), 'TOUT': (-4.5, 45), 'PRS': (688, 740), 'CO': (0, 10), 'NO': (0, 350), 'NO2': (0, 175), 'NOX': (0, 400), 'SO2': (0, 405), 'SR': (0, 1.2)}
}

# Límites universales para las que no cambian en el PDF
limites_universales = {'RH': (0, 100), 'WDR': (0, 360)}

# Aplicar filtros
for year, limites in rangos.items():
    mask_year = df['Year'] == year
    for col, (min_val, max_val) in limites.items():
        if col in df.columns:
            # Excepción especial del PDF: Omitir máximo de O3 en NTE2 en 2020
            if year == 2020 and col == 'O3':
                mask_nte2 = mask_year & (df['Estacion'] == 'NTE2') & (df[col] > 153)
                df.loc[mask_nte2, col] = np.nan
            
            # Anular lo que sea físicamente imposible
            fuera_de_rango = mask_year & ((df[col] < min_val) | (df[col] > max_val))
            df.loc[fuera_de_rango, col] = np.nan

# Aplicar universales
for col, (min_val, max_val) in limites_universales.items():
    fuera_de_rango = (df[col] < min_val) | (df[col] > max_val)
    df.loc[fuera_de_rango, col] = np.nan

os.makedirs("results", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

ESTACIONES_ESPERADAS = [
    "CE", "NE", "NE2", "NE3", "NO", "NO2", "NO3", "NTE", "NTE2",
    "SE", "SE2", "SE3", "SO", "SO2", "SUR",
]
faltantes = sorted(set(ESTACIONES_ESPERADAS) - set(df["Estacion"].unique()))
if faltantes:
    print(f"⚠️  Faltan estaciones completas en df: {faltantes}")

print("Calculando rachas de missingness por estación (grilla horaria completa)...")
inicio = df["Date"].min().floor("D")
fin = df["Date"].max().floor("D") + pd.Timedelta(hours=23)
full_idx = pd.date_range(inicio, fin, freq="h")

estaciones = sorted(df["Estacion"].unique())
rows = []
for est in estaciones:
    sub = df[df["Estacion"] == est].drop_duplicates(subset="Date").set_index("Date")
    sub = sub.reindex(full_idx)
    for col in cols_medicion:
        s = sub[col]
        na = s.isna()
        grp = (~na).cumsum()
        run_lengths = na.groupby(grp).sum()
        max_gap = int(run_lengths.max()) if len(run_lengths) > 0 else 0
        pct_valido = round(100 * (~na).mean(), 2)
        rows.append({
            "Estacion": est,
            "Variable": col,
            "pct_valido": pct_valido,
            "racha_max_horas": max_gap,
            "racha_max_dias": round(max_gap / 24, 1),
        })

diag = pd.DataFrame(rows)
diag.to_csv("results/diagnostico_rachas_15estaciones.csv", index=False)

print("\nTop 15 rachas más largas (Estacion x Variable):")
print(diag.sort_values("racha_max_horas", ascending=False).head(15).to_string(index=False))

print("\nNOX y PM2.5 por estación:")
print(
    diag[diag["Variable"].isin(["NOX", "PM2.5"])]
    .sort_values(["Variable", "racha_max_horas"], ascending=[True, False])
    .to_string(index=False)
)

# --- Heatmaps ---
for metric, cmap, label, fname in [
    ("racha_max_dias", "Reds", "Racha máxima sin dato (días)", "heatmap_rachas_estacion_variable.png"),
    ("pct_valido", "RdYlGn", "% de horas con dato válido", "heatmap_pct_valido_estacion_variable.png"),
]:
    pivot = diag.pivot(index="Estacion", columns="Variable", values=metric)[cols_medicion]
    vmin, vmax = (0, 100) if metric == "pct_valido" else (None, None)
    fig, ax = plt.subplots(figsize=(13, 6))
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=0.5, ax=ax, cbar_kws={"label": label})
    ax.set_title(f"{label} por estación y variable, 2020-2025", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"results/figures/{fname}")
    plt.close()
    print(f"  [OK] results/figures/{fname}")

# 4. SEPARACIÓN EN TABLAS INDIVIDUALES Y EXPORTACIÓN
print("\nSeparando y guardando bases de datos independientes...")
os.makedirs("data/processed/variables", exist_ok=True)

# Llevaremos un registro de cuántas filas válidas sobreviven por variable
resumen_filas = {}

for col in cols_medicion:
    # Nos quedamos con Date, Estacion y la variable específica
    df_var = df[['Date', 'Estacion', col]].copy()
    
    # Eliminamos las filas donde esta variable es NaN (datos limpios y reales 100%)
    df_var_limpia = df_var.dropna(subset=[col]).reset_index(drop=True)
    
    # Exportamos a Parquet
    ruta_salida = f"data/processed/variables/{col}_clean.parquet"
    df_var_limpia.to_parquet(ruta_salida, index=False)
    
    resumen_filas[col] = df_var_limpia.shape[0]
    print(f"✔️ {col} guardada. Filas válidas: {df_var_limpia.shape[0]:,}")

print("\n✅ ¡Fase 2 Completada! Tus variables están limpias, validadas y listas en 'data/processed/variables/'")

Iniciando Script 2: Limpieza, Rangos Dinámicos y Separación de Tablas...
Eliminando banderas de invalidación y ajustando tipos...
Aplicando rangos operativos del fabricante y del SIMA por año...
Calculando rachas de missingness por estación (grilla horaria completa)...

Top 15 rachas más largas (Estacion x Variable):
Estacion Variable  pct_valido  racha_max_horas  racha_max_dias
     NE3    PM2.5        4.20            50225          2092.7
     NO3    PM2.5        6.74            37094          1545.6
     NO3     PM10       44.76            25561          1065.0
     NO3       NO       36.60            25561          1065.0
     NO3       O3       42.97            25561          1065.0
     NO3       CO       41.85            25561          1065.0
     NO3      NOX       36.64            25561          1065.0
     NO3      NO2       36.65            25561          1065.0
     NO3      PRS       46.34            25561          1065.0
     NO3      WDR       50.35            25561     